# Лабораторная работа №4
## Модель однофазной многоканальной замкнутой системы обслуживания

Бирюкова Екатерина

ИУ5-81Б

## Постановка задачи

Рассматривается замкнутая система массового обслуживания, состоящая из:
- $k$ программистов, генерирующих задания;
- $m$ серверов, обрабатывающих задания.

После выполнения задания возможны два сценария:
- задание успешно завершается и приносит доход;
- задание может быть отозвано через случайное время ожидания.

Требуется:
1. Построить математическую модель системы.
2. Найти стационарные вероятности состояний.
3. Определить основные характеристики системы:
   - среднее число занятых серверов;
   - среднее число заявок в системе;
   - интенсивности потоков;
   - вероятность отказа (отзыва).
4. Рассчитать экономические показатели и определить оптимальное число серверов.

---

In [ ]:
m <- 6      # число серверов
k <- 14     # число программистов

t1 <- 11    # среднее время подготовки программы (мин)
t2 <- 8     # среднее время выполнения на сервере (мин)
t3 <- 6     # среднее время до отзыва программы (мин)

S1 <- 180   # доход за обслуженную заявку (руб)
S2 <- 3.5   # стоимость работы одного сервера (руб/мин)

In [ ]:
mu <- 1 / t2       # интенсивность обслуживания одним сервером
lambda <- 1 / t1   # интенсивность подготовки программы одним программистом
nu <- 1 / t3       # интенсивность отзыва ожидающей программы

cat("λ =", round(lambda, 5), "1/мин\n")
cat("μ =", round(mu, 5), "1/мин\n")
cat("ν =", round(nu, 5), "1/мин\n")

λ = 0.09091 1/мин
μ = 0.125 1/мин
ν = 0.16667 1/мин


## Описание состояний системы

Состояние системы задаётся числом программ $i$, находящихся в системе:
$$
i = 0, 1, 2, \dots, k
$$

Так как система замкнутая, в любой момент:
- $k - i$ программистов готовят новые программы;
- $\min(i, m)$ программ обслуживаются на серверах;
- $\max(i - m, 0)$ программ ожидают в очереди.

Переходы между состояниями:

- поступление новой программы:
$$
i \to i+1, \quad \lambda_i = (k-i)\lambda
$$

- уменьшение числа программ в системе возможно по двум причинам:
  1. завершение обслуживания;
  2. отзыв программы из очереди.

Поэтому для перехода
$$
i \to i-1
$$
интенсивность равна
$$
\mu_i^{(out)} = \min(i,m)\mu + \max(i-m,0)\nu
$$

где:
- $\mu = \frac{1}{t_2}$ — интенсивность обслуживания одной программы одним сервером;
- $\nu = \frac{1}{t_3}$ — интенсивность отзыва одной ожидающей программы.

### Граф состояний

$$
0 \;\mathop{\rightleftarrows}^{\lambda_0}_{\mu_1^{(out)}}\; 1
\;\mathop{\rightleftarrows}^{\lambda_1}_{\mu_2^{(out)}}\; 2
\;\mathop{\rightleftarrows}^{\lambda_2}_{\mu_3^{(out)}}\; \dots
\;\mathop{\rightleftarrows}^{\lambda_{k-1}}_{\mu_k^{(out)}}\; k
$$

In [ ]:
lambda_i <- function(i) {
  (k - i) * lambda
}

mu_serv_i <- function(i) {
  min(i, m) * mu
}

nu_i <- function(i) {
  max(i - m, 0) * nu
}

mu_out_i <- function(i) {
  mu_serv_i(i) + nu_i(i)
}

In [ ]:
states <- 0:k

transition_table <- data.frame(
  state = 0:k,
  lambda_i = c(sapply(0:(k - 1), lambda_i), NA),
  mu_service_i = c(NA, sapply(1:k, mu_serv_i)),
  nu_i = c(NA, sapply(1:k, nu_i)),
  mu_out_i = c(NA, sapply(1:k, mu_out_i))
)

transition_table

state,lambda_i,mu_service_i,nu_i,mu_out_i
<int>,<dbl>,<dbl>,<dbl>,<dbl>
0,1.27272727,NA,NA,NA
1,1.18181818,0.125,0.0000000,0.1250000
2,1.09090909,0.250,0.0000000,0.2500000
3,1.00000000,0.375,0.0000000,0.3750000
4,0.90909091,0.500,0.0000000,0.5000000
5,0.81818182,0.625,0.0000000,0.6250000
6,0.72727273,0.750,0.0000000,0.7500000
7,0.63636364,0.750,0.1666667,0.9166667
8,0.54545455,0.750,0.3333333,1.0833333


## Стационарные вероятности

Для замкнутой марковской системы стационарные вероятности состояний определяются рекуррентно:

$$
p_i = p_0 \prod_{j=1}^{i} \frac{\lambda_{j-1}}{\mu_j^{(out)}}
$$

где

$$
\mu_j^{(out)} = \min(j,m)\mu + \max(j-m,0)\nu
$$

а $p_0$ находится из условия нормировки:

$$
\sum_{i=0}^{k} p_i = 1
$$

In [ ]:
p <- numeric(k + 1)
p[1] <- 1

for (i in 1:k) {
  p[i + 1] <- p[i] * lambda_i(i - 1) / mu_out_i(i)
}

p <- p / sum(p)

prob_table <- data.frame(
  state = 0:k,
  p = round(p, 6)
)

prob_table

state,p
<int>,<dbl>
0,0.000496
1,0.005046
2,0.023853
3,0.069390
4,0.138780
5,0.201862
6,0.220213
7,0.174714
8,0.102629


In [ ]:
sum(p)

[1] 1

In [ ]:
L_sys <- sum((0:k) * p)

L_busy <- sum(sapply(0:k, function(i) min(i, m)) * p)

L_queue <- sum(sapply(0:k, function(i) max(i - m, 0)) * p)

mu_eff <- sum(sapply(0:k, mu_serv_i) * p)

nu_eff <- sum(sapply(0:k, nu_i) * p)

lambda_eff <- sum(sapply(0:(k - 1), lambda_i) * p[1:k])

arrival_state_prob <- sapply(0:(k - 1), lambda_i) * p[1:k]
arrival_state_prob <- arrival_state_prob / sum(arrival_state_prob)

if (m < k) {
  P_wait <- sum(arrival_state_prob[(m + 1):k])
} else {
  P_wait <- 0
}

P_withdraw <- nu_eff / lambda_eff

P_success <- mu_eff / lambda_eff

T_exit <- L_sys / lambda_eff

T_queue <- L_queue / lambda_eff

cat("Среднее число программ в системе:", round(L_sys, 4), "\n")
cat("Среднее число занятых серверов:", round(L_busy, 4), "\n")
cat("Среднее число программ в очереди:", round(L_queue, 4), "\n")
cat("Абсолютная пропускная способность:", round(mu_eff, 4), "прогр./мин\n")
cat("Интенсивность отзывов:", round(nu_eff, 4), "прогр./мин\n")
cat("Интенсивность поступления:", round(lambda_eff, 4), "прогр./мин\n")
cat("Вероятность, что программа не будет выполнена сразу:", round(P_wait, 4), "\n")
cat("Вероятность успешного завершения:", round(P_success, 4), "\n")
cat("Вероятность отзыва:", round(P_withdraw, 4), "\n")
cat("Среднее время пребывания программы в системе до выхода:", round(T_exit, 4), "мин\n")
cat("Среднее время ожидания в очереди до выхода:", round(T_queue, 4), "мин\n")
cat("Среднее время до получения результата теоретически здесь не вычисляется,\n")
cat("оно будет оценено отдельно по имитационной модели только для успешно завершённых программ.\n")

Среднее число программ в системе: 5.7805 
Среднее число занятых серверов: 5.1888 
Среднее число программ в очереди: 0.5917 
Абсолютная пропускная способность: 0.6486 прогр./мин
Интенсивность отзывов: 0.0986 прогр./мин
Интенсивность поступления: 0.7472 прогр./мин
Вероятность, что программа не будет выполнена сразу: 0.4736 
Вероятность успешного завершения: 0.868 
Вероятность отзыва: 0.132 
Среднее время пребывания программы в системе до выхода: 7.736 мин
Среднее время ожидания в очереди до выхода: 0.7919 мин
Среднее время до получения результата теоретически здесь не вычисляется,
оно будет оценено отдельно по имитационной модели только для успешно завершённых программ.


## Экономические показатели

Средняя прибыль в единицу времени вычисляется по формуле

$$
W(m) = S_1 \cdot \mu_{\text{eff}}(m) - S_2 \cdot m
$$

где:
- $S_1$ — доход от одной обслуженной программы;
- $\mu_{\text{eff}}(m)$ — абсолютная пропускная способность системы при $m$ серверах;
- $S_2$ — стоимость обслуживания одного сервера в минуту.

Оптимальным считается такое число серверов $m \ge 5$, при котором прибыль максимальна.

In [ ]:
W <- S1 * mu_eff - S2 * m

cat("Средняя прибыль системы:", round(W, 4), "руб/мин\n")

Средняя прибыль системы: 95.7479 руб/мин


## Поиск оптимального числа серверов

По условию задачи число серверов должно удовлетворять ограничению

$$
m \ge 5
$$

Поэтому перебираем все допустимые значения $m$ от 5 до $k$ и для каждого значения рассчитываем прибыль

$$
W(m) = S_1 \cdot \mu_{\text{eff}}(m) - S_2 \cdot m
$$

После этого выбираем значение $m$, при котором прибыль максимальна.

In [ ]:
calc_metrics <- function(m_val) {

  lambda_i_local <- function(i) {
    (k - i) * lambda
  }

  mu_serv_i_local <- function(i) {
    min(i, m_val) * mu
  }

  nu_i_local <- function(i) {
    max(i - m_val, 0) * nu
  }

  mu_out_i_local <- function(i) {
    mu_serv_i_local(i) + nu_i_local(i)
  }

  p_local <- numeric(k + 1)
  p_local[1] <- 1

  for (i in 1:k) {
    p_local[i + 1] <- p_local[i] * lambda_i_local(i - 1) / mu_out_i_local(i)
  }

  p_local <- p_local / sum(p_local)

  mu_eff_local <- sum(sapply(0:k, mu_serv_i_local) * p_local)
  nu_eff_local <- sum(sapply(0:k, nu_i_local) * p_local)
  lambda_eff_local <- sum(sapply(0:(k - 1), lambda_i_local) * p_local[1:k])
  L_queue_local <- sum(sapply(0:k, function(i) max(i - m_val, 0)) * p_local)

  arrival_state_prob_local <- sapply(0:(k - 1), lambda_i_local) * p_local[1:k]
  arrival_state_prob_local <- arrival_state_prob_local / sum(arrival_state_prob_local)

  if (m_val < k) {
    P_wait_local <- sum(arrival_state_prob_local[(m_val + 1):k])
  } else {
    P_wait_local <- 0
  }

  W_local <- S1 * mu_eff_local - S2 * m_val

  return(c(
    m = m_val,
    mu_eff = mu_eff_local,
    nu_eff = nu_eff_local,
    lambda_eff = lambda_eff_local,
    L_queue = L_queue_local,
    P_wait = P_wait_local,
    profit = W_local
  ))
}

m_values <- 5:k
metrics_list <- lapply(m_values, calc_metrics)
result <- as.data.frame(do.call(rbind, metrics_list))

result <- transform(
  result,
  m = as.integer(m),
  mu_eff = round(mu_eff, 4),
  nu_eff = round(nu_eff, 4),
  lambda_eff = round(lambda_eff, 4),
  L_queue = round(L_queue, 4),
  P_wait = round(P_wait, 4),
  profit = round(profit, 4)
)

result

best_row <- result[which.max(result$profit), ]

cat("Оптимальное число серверов:", best_row$m, "\n")
cat("Максимальная прибыль:", best_row$profit, "руб/мин\n")

m,mu_eff,nu_eff,lambda_eff,L_queue,P_wait,profit
<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
5,0.5760,0.1798,0.7558,1.0785,0.6817,86.1818
6,0.6486,0.0986,0.7472,0.5917,0.4736,95.7479
7,0.6959,0.0458,0.7417,0.2749,0.2728,100.7534
8,0.7213,0.0174,0.7387,0.1045,0.1258,101.8265
9,0.7322,0.0052,0.7374,0.0313,0.0448,100.2904
10,0.7358,0.0012,0.7370,0.0071,0.0118,97.4413
11,0.7367,0.0002,0.7369,0.0011,0.0022,94.1012
12,0.7368,0.0000,0.7368,0.0001,0.0002,90.6285
13,0.7368,0.0000,0.7368,0.0000,0.0000,87.1314


Оптимальное число серверов: 8 
Максимальная прибыль: 101.8265 руб/мин


In [ ]:
set.seed(42)

simulate_system <- function(m_val, T_max = 200000, T_warmup = 20000) {

  time <- 0
  state <- 0

  area_sys <- 0
  area_queue <- 0

  arrivals <- 0
  arrivals_wait <- 0
  served <- 0
  withdrawn <- 0

  total_time_in_system <- 0

  total_time_success <- 0

  jobs <- numeric(0)

  while (time < T_max) {
    lam <- (k - state) * lambda
    mu_serv <- min(state, m_val) * mu
    nu_out <- max(state - m_val, 0) * nu
    rate_sum <- lam + mu_serv + nu_out

    if (rate_sum == 0) break

    dt <- rexp(1, rate_sum)

    t1_seg <- time
    t2_seg <- min(time + dt, T_max)

    if (t2_seg > T_warmup) {
      left <- max(t1_seg, T_warmup)
      right <- t2_seg
      if (right > left) {
        area_sys <- area_sys + state * (right - left)
        area_queue <- area_queue + max(state - m_val, 0) * (right - left)
      }
    }

    time <- time + dt
    u <- runif(1) * rate_sum

    if (u < lam) {
      state <- state + 1
      jobs <- c(jobs, time)

      if (time > T_warmup) {
        arrivals <- arrivals + 1
        if (state - 1 >= m_val) {
          arrivals_wait <- arrivals_wait + 1
        }
      }

    } else if (u < lam + mu_serv) {
      if (length(jobs) > 0) {
        arrival_time <- jobs[1]
        jobs <- jobs[-1]

        if (arrival_time > T_warmup) {
          served <- served + 1
          total_time_in_system <- total_time_in_system + (time - arrival_time)
          total_time_success <- total_time_success + (time - arrival_time)
        }
      }

      state <- state - 1

    } else {
      if (length(jobs) > m_val) {
        withdraw_index <- m_val + 1
        arrival_time <- jobs[withdraw_index]
        jobs <- jobs[-withdraw_index]

        if (arrival_time > T_warmup) {
          withdrawn <- withdrawn + 1
          total_time_in_system <- total_time_in_system + (time - arrival_time)
        }
      }

      state <- state - 1
    }
  }

  obs_time <- T_max - T_warmup

  L_sys_exp <- area_sys / obs_time
  L_queue_exp <- area_queue / obs_time
  lambda_eff_exp <- arrivals / obs_time
  mu_eff_exp <- served / obs_time
  nu_eff_exp <- withdrawn / obs_time
  P_wait_exp <- arrivals_wait / arrivals

  T_exit_exp <- ifelse((served + withdrawn) > 0,
                       total_time_in_system / (served + withdrawn),
                       NA)

  T_success_exp <- ifelse(served > 0,
                          total_time_success / served,
                          NA)

  data.frame(
    m = m_val,
    L_sys_exp = round(L_sys_exp, 4),
    L_queue_exp = round(L_queue_exp, 4),
    lambda_eff_exp = round(lambda_eff_exp, 4),
    mu_eff_exp = round(mu_eff_exp, 4),
    nu_eff_exp = round(nu_eff_exp, 4),
    P_wait_exp = round(P_wait_exp, 4),
    T_exit_exp = round(T_exit_exp, 4),
    T_success_exp = round(T_success_exp, 4)
  )
}

exp_result <- simulate_system(m)
exp_result

m,L_sys_exp,L_queue_exp,lambda_eff_exp,mu_eff_exp,nu_eff_exp,P_wait_exp,T_exit_exp,T_success_exp
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
6,5.7841,0.5973,0.7464,0.648,0.0984,0.4761,7.7485,8.6296


In [ ]:
comparison <- data.frame(
  metric = c(
    "Вероятность ожидания",
    "Среднее время до выхода из системы",
    "Среднее число в очереди",
    "Абсолютная пропускная способность"
  ),
  theory = c(P_wait, T_exit, L_queue, mu_eff),
  experiment = c(
    exp_result$P_wait_exp,
    exp_result$T_exit_exp,
    exp_result$L_queue_exp,
    exp_result$mu_eff_exp
  )
)

comparison$theory <- round(comparison$theory, 4)
comparison$experiment <- round(comparison$experiment, 4)

comparison

cat("Оценка по имитации: среднее время до получения результата\n")
cat("(только для успешно завершённых программ) =",
    round(exp_result$T_success_exp, 4), "мин\n")

metric,theory,experiment
<chr>,<dbl>,<dbl>
Вероятность ожидания,0.4736,0.4761
Среднее время до выхода из системы,7.7360,7.7485
Среднее число в очереди,0.5917,0.5973
Абсолютная пропускная способность,0.6486,0.6480


Оценка по имитации: среднее время до получения результата
(только для успешно завершённых программ) = 8.6296 мин


## Вывод

В работе была построена модель замкнутой многоканальной системы массового обслуживания с конечным числом источников, несколькими серверами и возможностью отзыва программы из очереди.

Были получены стационарные вероятности состояний системы и на их основе вычислены основные характеристики:
- вероятность того, что поступившая программа не будет выполнена сразу;
- среднее время пребывания программы в системе до выхода из неё;
- среднее количество программ в очереди;
- абсолютная пропускная способность системы;
- вероятности успешного завершения и отзыва программы.

Также была найдена средняя прибыль системы и выполнен подбор оптимального числа серверов при ограничении $m \ge 5$.

Дополнительно теоретические результаты были проверены экспериментально с помощью имитационного моделирования. Экспериментальные значения оказались близки к теоретическим, что подтверждает корректность построенной модели.

Важно, что в теоретической части по формуле Литтла было найдено среднее время пребывания программы в системе до выхода из неё, то есть до успешного завершения или до отзыва. Отдельно в имитационной модели была оценена величина, более точно соответствующая формулировке задания: среднее время до получения пользователем результата, то есть среднее время только для успешно завершённых программ.